# 🔭 PANOSETI Run Preview

This notebook provides a convenience interface to preview the latest data products and configurations from a PANOSETI observing run.

In [ ]:
%matplotlib inline
import os
import glob
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pypff import PanosetiRun, PFFSequence, hkpff
from rich import print as rprint
from tqdm.notebook import tqdm

# --- Configuration ---
DATA_DIR = "/mnt/panoseti"  # Base directory containing .pffd runs
# -----------------------

## 1. Discover Latest Run

Automatically finding the most recent `.pffd` directory in the configured data path.

In [ ]:
def get_latest_run(base_dir):
    runs = sorted(glob.glob(os.path.join(base_dir, "*.pffd")))
    if not runs:
        return None
    return Path(runs[-1])

latest_run_path = get_latest_run(DATA_DIR)

if latest_run_path:
    print(f"✅ Latest Run Found: {latest_run_path.name}")
    run = PanosetiRun(latest_run_path)
    run.show()
else:
    print(f"❌ No runs found in {DATA_DIR}")

## 2. Time-Multiplexed Interleaving Analysis

Analyze the collection rate and scheduling of all discovered data products.

In [ ]:
def extract_timeline(run_obj, sample_rate=1000, precise=True):
    data = {}
    for prod_name in tqdm(run_obj.list_products(), desc="Scanning Timestamps"):
        seq = run_obj.get_product(prod_name)
        if len(seq) == 0: continue
        
        indices = list(range(0, len(seq), sample_rate))
        if indices[-1] != len(seq) - 1: indices.append(len(seq) - 1)
        
        times = np.array([seq.get_frame_time(i, precise=precise) / 1e9 for i in indices])
        data[prod_name] = {"times": times, "indices": np.array(indices)}
    return data

if 'run' in locals():
    sample_rate = 100
    timeline = extract_timeline(run, sample_rate)

    if timeline:
        t0 = min([d["times"].min() for d in timeline.values()])
        max_t = max([d["times"].max() for d in timeline.values()]) - t0
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        bin_size = 0.5
        bins = np.arange(0, max_t + bin_size, bin_size)
        colors = plt.cm.tab10.colors
        
        for i, (name, d) in enumerate(timeline.items()):
            norm_times = d["times"] - t0
            counts, _ = np.histogram(norm_times, bins=bins)
            fps = (counts / bin_size) * sample_rate
            color = colors[i % len(colors)]
            
            ax1.step(bins[:-1], fps, where='post', label=name, color=color, linewidth=2)
            ax1.fill_between(bins[:-1], fps, step='post', color=color, alpha=0.3)
            
            active_bins = bins[:-1][fps > 0]
            ax2.scatter(active_bins, [name] * len(active_bins), marker='s', s=50, color=color)

        ax1.set_title('Collection Rate (FPS)', fontweight='bold')
        ax1.set_ylabel('FPS')
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc='upper right', fontsize=8)
        
        ax2.set_title('Time-Multiplex Schedule', fontweight='bold')
        ax2.set_xlabel('Seconds since Run Start')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## 3. First N Frames Overview

Compact visualization of the start of each data product stream.

In [ ]:
if 'run' in locals():
    n_samples = 6
    products = run.list_products()
    if products:
        fig, axes = plt.subplots(len(products), n_samples, figsize=(2 * n_samples, 2.5 * len(products)), squeeze=False)

        for row_idx, name in enumerate(products):
            seq = run.get_product(name)
            count = min(n_samples, len(seq))
            
            for col_idx in range(n_samples):
                ax = axes[row_idx, col_idx]
                if col_idx < count:
                    header, data = seq.get_frame(col_idx)
                    img_data = data.reshape(seq.frame_config.image_shape)
                    ax.imshow(img_data, cmap='viridis', origin='lower')
                    if col_idx == 0:
                        ax.set_ylabel(name, fontsize=9, fontweight='bold')
                else:
                    ax.axis('off')
                
                ax.set_xticks([])
                ax.set_yticks([])
                if row_idx == 0:
                    ax.set_title(f"F{col_idx}", fontsize=10)

        plt.tight_layout()
        plt.show()
    else:
        print("No products found to plot.")

## 3. Interactive Data Product Explorer

Browse through images and pulse-height data with precise timestamps.

In [ ]:
if 'run' in locals():
    products = run.list_products()
    
    product_dropdown = widgets.Dropdown(
        options=products,
        description='Product:',
        layout=widgets.Layout(width='400px')
    )

    cmap_dropdown = widgets.Dropdown(
        options=['magma', 'viridis', 'plasma', 'inferno', 'gray', 'hot'],
        value='magma',
        description='Colormap:',
        layout=widgets.Layout(width='400px')
    )

    frame_slider = widgets.IntSlider(
        min=0,
        max=100,
        step=1,
        description='Frame:',
        continuous_update=False,
        layout=widgets.Layout(width='800px')
    )

    out = widgets.Output()

    def format_header(header):
        if hasattr(header, 'quabo_num'):
            return (
                f"Quabo {header.quabo_num} | Pkt: {header.pkt_num} | "
                f"TAI: {header.pkt_tai} | ns: {header.pkt_nsec}\n"
                f"UTC: {header.tv_sec}.{header.tv_usec:06d}"
            )
        elif hasattr(header, 'quabo_0'):
            return (
                f"Module Group | TAI (Q0): {header.quabo_0.pkt_tai} | ns: {header.quabo_0.pkt_nsec}\n"
                f"UTC: {header.quabo_0.tv_sec}.{header.quabo_0.tv_usec:06d}"
            )
        return str(header)

    def update_display(change=None):
        product_name = product_dropdown.value
        seq = run.get_product(product_name)
        frame_slider.max = len(seq) - 1
        
        frame_idx = min(frame_slider.value, len(seq) - 1)
        header, data = seq.get_frame(frame_idx)
        precise_time_ns = seq.get_frame_time(frame_idx, precise=True)
        
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 6))
            
            # Force 2D visualization for all products
            # Image shape is retrieved from sequence frame_config
            img_data = data.reshape(seq.frame_config.image_shape)
            
            im = ax.imshow(img_data, cmap=cmap_dropdown.value, origin='lower', interpolation='nearest')
            fig.colorbar(im, ax=ax, label='ADC Value')
            
            title = f"{product_name} [Frame {frame_idx}]\n{format_header(header)}\nPrecise: {precise_time_ns} ns"
            ax.set_title(title, loc='left', family='monospace', fontsize=10)
            plt.tight_layout()
            plt.show()

    product_dropdown.observe(update_display, names='value')
    cmap_dropdown.observe(update_display, names='value')
    frame_slider.observe(update_display, names='value')
    update_display()

    display(widgets.VBox([widgets.HBox([product_dropdown, cmap_dropdown]), frame_slider, out]))

## 4. Configuration Viewer

Inspect run configurations with Rich formatting.

In [ ]:
if 'run' in locals():
    config_select = widgets.Dropdown(
        options=sorted(run.configs.keys()),
        description='Config:',
        layout=widgets.Layout(width='400px')
    )
    
    config_out = widgets.Output()
    
    def show_config(change=None):
        with config_out:
            clear_output(wait=True)
            cfg = run.configs[config_select.value]
            if hasattr(cfg, 'model_dump'):
                rprint(cfg.model_dump())
            else:
                rprint(cfg)
                
    config_select.observe(show_config, names='value')
    show_config()
    display(widgets.VBox([config_select, config_out]))

## 5. Log File Viewer

Read standard and JSONL logs directly from the run directory.

In [ ]:
if 'run' in locals():
    available_logs = run.list_logs() + list(run.metadata.keys())
    
    log_select = widgets.Dropdown(
        options=sorted(available_logs),
        description='Log File:',
        layout=widgets.Layout(width='500px')
    )
    
    log_out = widgets.Output()
    
    def show_log(change=None):
        name = log_select.value
        with log_out:
            clear_output(wait=True)
            if name in run.metadata:
                # JSONL metadata (parsed)
                rprint(run.metadata[name][:20]) # Show first 20 entries
                print(f"... [{len(run.metadata[name])} total entries]")
            else:
                # Standard text log
                print(run.get_log(name))
                
    log_select.observe(show_log, names='value')
    show_log()
    display(widgets.VBox([log_select, log_out]))